# MAS1 – Analýza sítě: fb-pages-sport
## Část 2a – Základní strukturální vlastnosti

**Data:** [fb-pages-sport](https://networkrepository.com/fb-pages-sport.php) z Network Data Repository (Rossi & Ahmed, 2015).
Vrcholy jsou ověřené (blue verified) facebookové stránky sportovců, hrana znamená, že se dvě stránky navzájem označily „To se mi líbí“.
Síť je neorientovaná a bez vah, data byla sesbírána v listopadu 2017. ID vrcholů jsou anonymizovaná.

**Potřebné knihovny:** `pip install networkx igraph pandas matplotlib scipy`

`igraph` (jádro v C) se používá jen tam, kde by NetworkX bylo na síti s ~14 tisíci vrcholy příliš pomalé:
closeness a betweenness centralita a rozložení (layout) pro vizualizaci. Vše ostatní je v NetworkX nebo vlastní implementaci jako na cvičeních.

In [ ]:
import random
from collections import Counter
from pathlib import Path

import igraph as ig
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from matplotlib.collections import LineCollection
from matplotlib.colors import LogNorm

SOUBOR = "fb-pages-sport.edges"   # upravte podle názvu souboru v rozbaleném zipu (.edges / .csv / .mtx)
VYSTUP = Path("vystupy")          # sem se ukládají obrázky a tabulky pro PDF
VYSTUP.mkdir(exist_ok=True)
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
ig.set_random_number_generator(random)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight"})
pd.set_option("display.float_format", lambda x: f"{x:.6g}")

### Načtení dat
Načítání je upravené z cvičení tak, aby zvládlo formáty z networkrepository: oddělovač čárka, středník, mezera nebo tabulátor,
komentáře začínající `%` nebo `#` a u formátu `.mtx` i úvodní řádek s rozměry matice. Smyčky a duplicitní hrany se odstraní.

In [ ]:
def load_edges(filename):
    edges = []
    is_mtx = str(filename).endswith(".mtx")
    header_skipped = False

    with open(filename, encoding="utf-8-sig") as f:
        for line in f:
            line = line.strip()
            if not line or line[0] in "%#":
                continue
            if is_mtx and not header_skipped:   # radek "pocet_radku pocet_sloupcu pocet_hran"
                header_skipped = True
                continue
            values = line.replace(",", " ").replace(";", " ").split()
            if len(values) < 2:
                continue
            try:
                u, v = int(values[0]), int(values[1])
            except ValueError:                  # textova hlavicka
                continue
            edges.append((u, v))

    return edges


edges = load_edges(SOUBOR)
self_loops = sum(1 for u, v in edges if u == v)

G = nx.Graph()
G.add_edges_from((u, v) for u, v in edges if u != v)

print(f"Načtených řádků s hranou: {len(edges)}")
print(f"Odstraněné smyčky:        {self_loops}")
print(f"Odstraněné duplicity:     {len(edges) - self_loops - G.number_of_edges()}")

## 2a-i. Počet vrcholů, hran, hustota

In [ ]:
n = G.number_of_nodes()
m = G.number_of_edges()
hustota = 2 * m / (n * (n - 1))

assert abs(hustota - nx.density(G)) < 1e-12   # kontrola vlastniho vypoctu

tab_zakladni = pd.DataFrame({
    "vlastnost": ["Počet vrcholů", "Počet hran", "Hustota"],
    "hodnota": [n, m, hustota],
})
tab_zakladni

**Co sledovat:** hustota říká, jaký podíl ze všech možných hran v síti skutečně existuje.
U reálných sítí bývá velmi malá (řídké sítě), každá stránka je propojená jen se zlomkem ostatních.

## 2a-ii. Průměrný stupeň, maximální stupeň, distribuce stupňů

In [ ]:
degrees = dict(G.degree())
deg_values = np.array(list(degrees.values()))

d_min, d_max, d_avg = deg_values.min(), deg_values.max(), deg_values.mean()

tab_stupne = pd.DataFrame({
    "vlastnost": ["Minimální stupeň", "Maximální stupeň", "Průměrný stupeň", "Medián stupně", "Směrodatná odchylka"],
    "hodnota": [d_min, d_max, d_avg, np.median(deg_values), deg_values.std()],
})
tab_stupne

In [ ]:
# CETNOST A RELATIVNI CETNOST (jako na cviceni)
frequency = Counter(deg_values)
tab_cetnost = pd.DataFrame({
    "stupen": sorted(frequency),
    "cetnost": [frequency[d] for d in sorted(frequency)],
})
tab_cetnost["rel_cetnost"] = tab_cetnost["cetnost"] / n
tab_cetnost.to_csv(VYSTUP / "distribuce_stupnu.csv", index=False)

tab_cetnost.head(15)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].bar(tab_cetnost["stupen"], tab_cetnost["cetnost"], width=1.0, color="steelblue")
axes[0].axvline(d_avg, color="red", linestyle="--", label=f"průměr = {d_avg:.2f}")
axes[0].set_xlabel("Stupeň vrcholu k")
axes[0].set_ylabel("Počet vrcholů")
axes[0].set_title("Distribuce stupňů (lineární osy)")
axes[0].legend()

axes[1].scatter(tab_cetnost["stupen"], tab_cetnost["rel_cetnost"], s=12, color="steelblue", edgecolor="black", linewidth=0.3)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("Stupeň vrcholu k (log)")
axes[1].set_ylabel("P(k) (log)")
axes[1].set_title("Distribuce stupňů (log-log)")
axes[1].grid(True, which="both", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.savefig(VYSTUP / "distribuce_stupnu.png")
plt.show()

**Co sledovat:** pokud je distribuce silně zešikmená (většina vrcholů má malý stupeň, několik málo vrcholů obrovský)
a v log-log grafu klesá přibližně lineárně, jde o síť s „těžkým chvostem“, typickou pro sociální sítě (blízko bezškálové síti).
Porovnejte maximální stupeň s průměrem: velký rozdíl znamená existenci výrazných hubů, tedy velmi populárních sportovců.

## 2a-iii. Centrality
Počítáme čtyři centrality, všechny normalizované:

- **Stupňová** – podíl ostatních vrcholů, se kterými je vrchol přímo spojen (popularita).
- **Closeness** – jak blízko má vrchol v průměru ke všem ostatním. Síť nemusí být souvislá, proto je použita
  korekce Wasserman–Faust (stejná jako výchozí v NetworkX), aby vrcholy v malých komponentách nevycházely uměle vysoko.
- **Betweenness** – na kolika nejkratších cestách mezi dvojicemi vrcholů vrchol leží (role „mostu“ mezi skupinami).
- **Eigenvector** – vrchol je důležitý, pokud je spojený s jinými důležitými vrcholy. Pro nesouvislou síť není jednoznačně
  definovaná, proto se počítá v největší komponentě a vrcholy v malých komponentách mají hodnotu 0.

Closeness a betweenness se počítají přesně přes igraph (řádově jednotky minut). Výsledek se uloží do CSV,
takže při dalším spuštění notebooku se už jen načte. Pokud změníte vstupní data, smažte `vystupy/centrality.csv`.

In [ ]:
def compute_centralities(G):
    nodes = list(G.nodes())
    index_of = {v: i for i, v in enumerate(nodes)}
    n = len(nodes)

    g = ig.Graph(n=n, edges=[(index_of[u], index_of[v]) for u, v in G.edges()], directed=False)

    # velikost komponenty kazdeho vrcholu (pro korekci closeness)
    comps = g.connected_components()
    sizes = comps.sizes()
    comp_size = np.array([sizes[c] for c in comps.membership])

    stupnova = np.array(g.degree()) / (n - 1)

    closeness = np.nan_to_num(np.array(g.closeness(), dtype=float))   # v ramci komponenty
    closeness = closeness * (comp_size - 1) / (n - 1)                   # korekce Wasserman-Faust

    betweenness = np.array(g.betweenness(directed=False)) / ((n - 1) * (n - 2) / 2)

    # eigenvector neni pro nesouvislou sit jednoznacna -> pocita se v nejvetsi komponente, ostatni vrcholy = 0
    giant_nodes = max(nx.connected_components(G), key=len)
    eig = nx.eigenvector_centrality_numpy(G.subgraph(giant_nodes))
    eigenvector = np.array([abs(eig.get(v, 0.0)) for v in nodes])

    return pd.DataFrame({
        "stupnova": stupnova,
        "closeness": closeness,
        "betweenness": betweenness,
        "eigenvector": eigenvector,
    }, index=pd.Index(nodes, name="vrchol"))


CENTR_FILE = VYSTUP / "centrality.csv"

if CENTR_FILE.exists() and len(pd.read_csv(CENTR_FILE)) == n:
    centrality = pd.read_csv(CENTR_FILE, index_col="vrchol")
    print("Centrality načteny z", CENTR_FILE)
else:
    centrality = compute_centralities(G)
    centrality.to_csv(CENTR_FILE)
    print("Centrality spočítány a uloženy do", CENTR_FILE)

centrality.describe().T[["mean", "50%", "max"]].rename(columns={"mean": "průměr", "50%": "medián", "max": "maximum"})

In [ ]:
# TOP 10 vrcholu podle kazde centrality
top10 = pd.concat(
    {name: centrality[name].nlargest(10).reset_index().set_axis(["vrchol", "hodnota"], axis=1)
     for name in centrality.columns},
    axis=1,
)
top10.index = range(1, 11)
top10.to_csv(VYSTUP / "top10_centrality.csv")
top10

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, name in zip(axes.flat, centrality.columns):
    ax.hist(centrality[name], bins=60, color="steelblue", edgecolor="black", linewidth=0.3)
    ax.set_yscale("log")
    ax.set_title(f"Distribuce: {name} centralita")
    ax.set_xlabel("hodnota centrality")
    ax.set_ylabel("počet vrcholů (log)")

plt.tight_layout()
plt.savefig(VYSTUP / "distribuce_centralit.png")
plt.show()

In [ ]:
# Jak moc se centrality shoduji v poradi vrcholu (Spearmanova korelace)
centrality.corr(method="spearman")

**Co sledovat:**
- Jsou nejvýše postavené vrcholy u různých centralit stejné? Vrchol s vysokou betweenness, ale průměrným stupněm,
  je typicky „most“ mezi skupinami (např. stránka, která propojuje dva sporty nebo dvě země).
- Distribuce betweenness bývá extrémně zešikmená: drtivá většina vrcholů leží na zanedbatelném počtu nejkratších cest.
- Vysoká korelace stupně a eigenvector centrality ukazuje, že populární stránky jsou propojené hlavně mezi sebou.

## 2a-iv. Shlukovací koeficient a shlukovací efekt
Shlukovací koeficient vrcholu je spočítán vlastní funkcí stejně jako na cvičení (počet hran mezi sousedy / maximální možný počet),
jen s množinami sousedů místo matice, aby výpočet byl rychlý i pro velkou síť. Výsledek je ověřen proti NetworkX.

In [ ]:
def clustering_coefficients(G):
    neighbours = {v: set(G[v]) for v in G}
    coefficients, links = {}, {}

    for v, N in neighbours.items():
        k = len(N)
        e = sum(len(N & neighbours[u]) for u in N) // 2   # hrany mezi sousedy (= trojuhelniky pres v)
        links[v] = e
        coefficients[v] = 2 * e / (k * (k - 1)) if k >= 2 else 0.0

    return coefficients, links


clustering, links = clustering_coefficients(G)

avg_cc = np.mean(list(clustering.values()))
transitivity = sum(links.values()) / sum(d * (d - 1) / 2 for d in deg_values)   # 3*trojuhelniky / trojice
pocet_trojuhelniku = sum(links.values()) // 3

assert abs(avg_cc - nx.average_clustering(G)) < 1e-9
assert abs(transitivity - nx.transitivity(G)) < 1e-9

tab_cc = pd.DataFrame({
    "vlastnost": ["Průměrný shlukovací koeficient", "Globální tranzitivita", "Počet trojúhelníků",
                  "Očekávaný CC náhodného grafu (= hustota)"],
    "hodnota": [avg_cc, transitivity, pocet_trojuhelniku, hustota],
})
tab_cc

In [ ]:
# SHLUKOVACI EFEKT – prumerny CC pro vrcholy daneho stupne
cc_by_degree = (
    pd.DataFrame({"stupen": pd.Series(degrees), "cc": pd.Series(clustering)})
    .groupby("stupen")["cc"].agg(prumerny_cc="mean", pocet_vrcholu="size")
    .reset_index()
)
cc_by_degree.to_csv(VYSTUP / "shlukovaci_efekt.csv", index=False)
cc_by_degree.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].scatter(cc_by_degree["stupen"], cc_by_degree["prumerny_cc"], s=14, color="steelblue", edgecolor="black", linewidth=0.3)
axes[0].axhline(avg_cc, color="red", linestyle="--", label=f"průměrný CC = {avg_cc:.3f}")
axes[0].set_xscale("log")
axes[0].set_ylim(-0.02, 1.02)
axes[0].set_xlabel("Stupeň k (log)")
axes[0].set_ylabel("Průměrný CC")
axes[0].set_title("Shlukovací efekt")
axes[0].legend()
axes[0].grid(True, linestyle=":", alpha=0.5)

nenulove = cc_by_degree[cc_by_degree["prumerny_cc"] > 0]
axes[1].scatter(nenulove["stupen"], nenulove["prumerny_cc"], s=14, color="steelblue", edgecolor="black", linewidth=0.3)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("Stupeň k (log)")
axes[1].set_ylabel("Průměrný CC (log)")
axes[1].set_title("Shlukovací efekt (log-log)")
axes[1].grid(True, which="both", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.savefig(VYSTUP / "shlukovaci_efekt.png")
plt.show()

**Co sledovat:**
- Porovnejte průměrný CC s hustotou. V náhodném grafu by CC odpovídal zhruba hustotě; pokud je reálný CC o několik řádů vyšší,
  síť má výrazné lokální shlukování („přátelé mých přátel jsou i moji přátelé“), což naznačuje komunitní strukturu.
- Pokud průměrný CC s rostoucím stupněm klesá (v log-log grafu přibližně po přímce), vrcholy s malým stupněm jsou součástí
  hustých skupinek, zatímco huby propojují různé skupiny, které spolu jinak nesouvisí (hierarchická struktura sítě).

## 2a-v. Souvislost – počet komponent a distribuce jejich velikostí

In [ ]:
components = sorted(nx.connected_components(G), key=len, reverse=True)
comp_sizes = [len(c) for c in components]
giant = components[0]

tab_komponenty = pd.DataFrame({
    "vlastnost": ["Počet souvislých komponent", "Velikost největší komponenty",
                  "Podíl vrcholů v největší komponentě", "Velikost 2. největší komponenty"],
    "hodnota": [len(components), len(giant), len(giant) / n, comp_sizes[1] if len(comp_sizes) > 1 else 0],
})

size_freq = Counter(comp_sizes)
tab_velikosti = pd.DataFrame({"velikost_komponenty": sorted(size_freq),
                              "pocet_komponent": [size_freq[s] for s in sorted(size_freq)]})
tab_velikosti.to_csv(VYSTUP / "velikosti_komponent.csv", index=False)

display(tab_komponenty)
display(tab_velikosti)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(tab_velikosti["velikost_komponenty"], tab_velikosti["pocet_komponent"],
           s=50, color="steelblue", edgecolor="black", zorder=3)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Velikost komponenty (počet vrcholů, log)")
ax.set_ylabel("Počet komponent (log)")
ax.set_title("Distribuce velikostí souvislých komponent")
ax.grid(True, which="both", linestyle=":", alpha=0.5)
ax.annotate(f"největší komponenta\n{len(giant)} vrcholů", xy=(len(giant), 1), xytext=(0.55, 0.55),
            textcoords="axes fraction", arrowprops=dict(arrowstyle="->"))
plt.tight_layout()
plt.savefig(VYSTUP / "velikosti_komponent.png")
plt.show()

**Co sledovat:** typická reálná síť má jednu obří komponentu, která obsahuje drtivou většinu vrcholů, a několik
malých izolovaných skupinek (dvojice, trojice stránek, které se lajkují jen navzájem). Popište, jak velký podíl
tvoří obří komponenta a co mohou malé komponenty představovat.

## 2a-vi. Vizualizace se zvýrazněním hubů
Kreslí se největší komponenta (malé komponenty by se jinak rozházely po okrajích a nic by nepřinesly).
Rozložení se počítá algoritmem Fruchterman–Reingold v igraph (v NetworkX by trvalo velmi dlouho).
Velikost a barva vrcholu odpovídá centralitě, 10 nejvýznamnějších vrcholů je zvýrazněno červeně s ID.

Pro hezčí vizualizaci do práce se navíc exportuje soubor `graf.gexf` pro Gephi se všemi spočítanými vlastnostmi
(stačí otevřít, spustit ForceAtlas 2 a velikost/barvu nastavit podle sloupců).

In [ ]:
H = G.subgraph(giant).copy()
h_nodes = list(H.nodes())
h_index = {v: i for i, v in enumerate(h_nodes)}

gH = ig.Graph(n=len(h_nodes), edges=[(h_index[u], h_index[v]) for u, v in H.edges()], directed=False)
layout = gH.layout_fruchterman_reingold(niter=1000)
pos = {v: layout[i] for i, v in enumerate(h_nodes)}

In [ ]:
def draw_network(ax, H, pos, values, title, top_k=10, cmap="viridis"):
    # hrany najednou pres LineCollection (rychle i pro desitky tisic hran)
    segments = [(pos[u], pos[v]) for u, v in H.edges()]
    ax.add_collection(LineCollection(segments, colors="grey", linewidths=0.15, alpha=0.12, zorder=1))

    # vrcholy serazene podle hodnoty, aby huby byly navrchu
    order = sorted(H.nodes(), key=lambda v: values[v])
    xy = np.array([pos[v] for v in order])
    vals = np.array([values[v] for v in order])
    sizes = 1 + 120 * (vals / vals.max())

    positive = vals[vals > 0]
    norm = LogNorm(vmin=positive.min(), vmax=vals.max()) if len(positive) else None
    sc = ax.scatter(xy[:, 0], xy[:, 1], s=sizes, c=np.clip(vals, positive.min(), None), cmap=cmap,
                    norm=norm, linewidths=0, zorder=2)

    hubs = sorted(H.nodes(), key=lambda v: values[v], reverse=True)[:top_k]
    hub_xy = np.array([pos[v] for v in hubs])
    ax.scatter(hub_xy[:, 0], hub_xy[:, 1], s=160, facecolor="red", edgecolor="black", linewidths=0.8, zorder=3)
    for v in hubs:
        ax.annotate(str(v), pos[v], xytext=(4, 4), textcoords="offset points", fontsize=8,
                    fontweight="bold", zorder=4,
                    bbox=dict(boxstyle="round,pad=0.15", facecolor="white", alpha=0.7, linewidth=0))

    ax.set_title(title)
    ax.autoscale()
    ax.set_aspect("equal")
    ax.axis("off")
    return sc


fig, axes = plt.subplots(1, 2, figsize=(18, 9))

sc0 = draw_network(axes[0], H, pos, degrees,
                   "Největší komponenta – velikost a barva podle stupně\n(červeně 10 vrcholů s nejvyšším stupněm)")
fig.colorbar(sc0, ax=axes[0], shrink=0.6, label="stupeň (log)")

btw = centrality["betweenness"].to_dict()
sc1 = draw_network(axes[1], H, pos, btw,
                   "Největší komponenta – velikost a barva podle betweenness\n(červeně 10 vrcholů s nejvyšší betweenness)",
                   cmap="plasma")
fig.colorbar(sc1, ax=axes[1], shrink=0.6, label="betweenness (log)")

plt.tight_layout()
plt.savefig(VYSTUP / "vizualizace_hubu.png", dpi=250)
plt.show()

In [ ]:
# EXPORT DO GEPHI (GEXF) se vsemi lokalnimi vlastnostmi
membership = {v: i for i, comp in enumerate(components) for v in comp}

G_export = nx.Graph()
for v in G.nodes():
    G_export.add_node(v,
        label=str(v),
        stupen=int(degrees[v]),
        closeness=float(centrality.at[v, "closeness"]),
        betweenness=float(centrality.at[v, "betweenness"]),
        eigenvector=float(centrality.at[v, "eigenvector"]),
        shlukovaci_koeficient=float(clustering[v]),
        komponenta=int(membership[v]),
    )
G_export.add_edges_from(G.edges())
nx.write_gexf(G_export, VYSTUP / "graf.gexf")

# lokalni vlastnosti do CSV (jako na cviceni)
mereni = centrality.copy()
mereni.insert(0, "stupen", pd.Series(degrees))
mereni["shlukovaci_koeficient"] = pd.Series(clustering)
mereni["komponenta"] = pd.Series(membership)
mereni.to_csv(VYSTUP / "mereni.csv")

print("Uloženo: vystupy/graf.gexf, vystupy/mereni.csv")

**Co sledovat:** jsou huby (podle stupně) ve středu hustého jádra, nebo rozptýlené? Leží vrcholy s nejvyšší betweenness
na rozhraní mezi hustšími shluky? Pokud se nejvyšší stupeň a nejvyšší betweenness liší, ukažte to na obrázku:
první skupina jsou „populární“ stránky, druhá „propojovatelé“ mezi skupinami.

## Souhrnná tabulka globálních vlastností (do PDF)

In [ ]:
souhrn = pd.concat([tab_zakladni, tab_stupne, tab_cc, tab_komponenty], ignore_index=True)
souhrn.to_csv(VYSTUP / "souhrn_globalnich_vlastnosti.csv", index=False)
souhrn